In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [11]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [16]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/kaviya7/nlp-disaster/sample_submission.csv
/kaggle/input/datasets/kaviya7/nlp-disaster/train.csv
/kaggle/input/datasets/kaviya7/nlp-disaster/test.csv


In [18]:
DATA_PATH = "/kaggle/input/datasets/kaviya7/nlp-disaster"

train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")
sample_submission = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Submission shape:", sample_submission.shape)

train_df.head()

Train shape: (7613, 5)
Test shape: (3263, 4)
Submission shape: (3263, 2)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [19]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

model_df = train_df[
    ["id", "keyword", "location", "text", "target"]
].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        model_df,
        y=model_df["target"],
        groups=model_df["text"]
    )
)

train_data = model_df.iloc[train_idx].copy()
val_data = model_df.iloc[val_idx].copy()

print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))

Training samples: 6083
Validation samples: 1530


In [20]:
overlap = set(train_data["text"]).intersection(
    set(val_data["text"])
)

print("Identical text overlap:", len(overlap))

Identical text overlap: 0


In [21]:
print("Training target counts:")
print(train_data["target"].value_counts().sort_index())

print("\nValidation target counts:")
print(val_data["target"].value_counts().sort_index())

Training target counts:
target
0    3488
1    2595
Name: count, dtype: int64

Validation target counts:
target
0    854
1    676
Name: count, dtype: int64


In [22]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully.


In [23]:
sample_text = train_data["text"].iloc[0]

encoded_sample = tokenizer(
    sample_text,
    truncation=True,
    max_length=128
)

print("Original:")
print(sample_text)

print("\nTokens:")
print(tokenizer.tokenize(sample_text))

print("\nToken IDs:")
print(encoded_sample["input_ids"])

Original:
Forest fire near La Ronge Sask. Canada

Tokens:
['forest', 'fire', 'near', 'la', 'ron', '##ge', 'sas', '##k', '.', 'canada']

Token IDs:
[101, 3224, 2543, 2379, 2474, 6902, 3351, 21871, 2243, 1012, 2710, 102]


In [24]:
token_lengths = train_df["text"].apply(
    lambda text: len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
)

print(token_lengths.describe())

print("\nMaximum token length:", token_lengths.max())
print(
    "Tweets longer than 128 tokens:",
    (token_lengths > 128).sum()
)

count    7613.000000
mean       33.108761
std        12.142242
min         3.000000
25%        25.000000
50%        33.000000
75%        42.000000
max        84.000000
Name: text, dtype: float64

Maximum token length: 84
Tweets longer than 128 tokens: 0


In [25]:
from torch.utils.data import Dataset

class DisasterTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = int(self.labels.iloc[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [26]:
train_dataset = DisasterTweetDataset(
    texts=train_data["text"],
    labels=train_data["target"],
    tokenizer=tokenizer,
    max_length=128
)

val_dataset = DisasterTweetDataset(
    texts=val_data["text"],
    labels=val_data["target"],
    tokenizer=tokenizer,
    max_length=128
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 6083
Validation dataset: 1530


In [27]:
sample = train_dataset[0]

print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label:", sample["labels"])

Input IDs shape: torch.Size([128])
Attention mask shape: torch.Size([128])
Label: tensor(1)


In [28]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to("cuda")

print("Model loaded on:", next(model.parameters()).device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on: cuda:0


In [29]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions),
        "f1": f1_score(labels, predictions)
    }

In [30]:
from transformers import TrainingArguments, Trainer

In [31]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert-disaster",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,

    report_to="none",

    seed=SEED
)

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

In [33]:
train_result = trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.817594,0.821297,0.824183,0.812596,0.782544,0.797287
2,0.693967,0.829765,0.832680,0.842020,0.764793,0.801550
3,0.551321,0.867988,0.825490,0.820031,0.775148,0.796958


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


In [34]:
transformer_metrics = trainer.evaluate()

transformer_metrics

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.8297724723815918,
 'eval_accuracy': 0.8326797385620915,
 'eval_precision': 0.8420195439739414,
 'eval_recall': 0.764792899408284,
 'eval_f1': 0.8015503875968992,
 'eval_runtime': 3.8112,
 'eval_samples_per_second': 401.447,
 'eval_steps_per_second': 6.297,
 'epoch': 3.0}

In [35]:
print("DistilBERT Validation Results")
print("-" * 35)

print(f"Accuracy : {transformer_metrics['eval_accuracy']:.4f}")
print(f"Precision: {transformer_metrics['eval_precision']:.4f}")
print(f"Recall   : {transformer_metrics['eval_recall']:.4f}")
print(f"F1 Score : {transformer_metrics['eval_f1']:.4f}")

DistilBERT Validation Results
-----------------------------------
Accuracy : 0.8327
Precision: 0.8420
Recall   : 0.7648
F1 Score : 0.8016


In [36]:
transformer_output = trainer.predict(val_dataset)

transformer_predictions = np.argmax(
    transformer_output.predictions,
    axis=-1
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [37]:
transformer_cm = confusion_matrix(
    val_data["target"],
    transformer_predictions
)

transformer_cm_df = pd.DataFrame(
    transformer_cm,
    index=["Actual Non-Disaster", "Actual Disaster"],
    columns=["Predicted Non-Disaster", "Predicted Disaster"]
)

transformer_cm_df

,Predicted Non-Disaster,Predicted Disaster
Actual Non-Disaster,757,97
Actual Disaster,159,517


In [38]:
transformer_errors = val_data[
    ["id", "text", "target"]
].copy()

transformer_errors["prediction"] = transformer_predictions

transformer_errors["error_type"] = "Correct"

transformer_errors.loc[
    (transformer_errors["target"] == 1) &
    (transformer_errors["prediction"] == 0),
    "error_type"
] = "False Negative"

transformer_errors.loc[
    (transformer_errors["target"] == 0) &
    (transformer_errors["prediction"] == 1),
    "error_type"
] = "False Positive"

transformer_errors["error_type"].value_counts()

error_type
Correct           1274
False Negative     159
False Positive      97
Name: count, dtype: int64

In [39]:
transformer_errors[
    transformer_errors["error_type"] == "False Positive"
][["text", "target", "prediction"]].head(10)

,text,target,prediction
35,On plus side LOOK AT THE SKY LAST NIGHT IT WAS...,0,1
58,They sky was ablaze tonight in Los Angeles. I'...,0,1
80,mom: 'we didn't get home as fast as we wished'...,0,1
380,Two Jewish Terrorists Charged In Historic-Chur...,0,1
397,Mourning notices for stabbing arson victims st...,0,1
423,#Vegetarian #Vegan Video shows arsonist torchi...,0,1
430,Arsonists being blamed for a blaze at a plasti...,0,1
589,RT alisonannyoung: EXCLUSIVE: FedEx no longer ...,0,1
591,#world FedEx no longer to transport bioterror ...,0,1
1232,Fire hazard associated with installation of no...,0,1


In [40]:
transformer_errors[
    transformer_errors["error_type"] == "False Negative"
][["text", "target", "prediction"]].head(10)

,text,target,prediction
7,I'm on top of the hill and I can see a fire in...,1,0
229,Ready to get annihilated for the BUCS game,1,0
234,@TomcatArts thus explaining why you were all a...,1,0
247,annihilating quarterstaff of annihilation,1,0
249,:StarMade: :Stardate 3: :Planetary Annihilatio...,1,0
251,U.S National Park Services Tonto National Fore...,1,0
253,U.S National Park Services Tonto National Fore...,1,0
269,World Annihilation vs Self Transformation http...,1,0
301,Short Reading\n\nApocalypse 21:1023 \n\nIn the...,1,0
354,Salvation Army hosts rally to reconnect father...,1,0


In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_compare = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000
)

X_train_compare = tfidf_compare.fit_transform(train_data["text"])
X_val_compare = tfidf_compare.transform(val_data["text"])

lr_compare = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED
)

lr_compare.fit(
    X_train_compare,
    train_data["target"]
)

tfidf_predictions = lr_compare.predict(X_val_compare)

In [43]:
print(
    "TF-IDF F1:",
    f1_score(val_data["target"], tfidf_predictions)
)

TF-IDF F1: 0.7659894657637322


In [44]:
comparison_errors = val_data[
    ["text", "target"]
].copy()

comparison_errors["tfidf_prediction"] = tfidf_predictions
comparison_errors["distilbert_prediction"] = transformer_predictions

transformer_fixed = comparison_errors[
    (comparison_errors["tfidf_prediction"] != comparison_errors["target"]) &
    (comparison_errors["distilbert_prediction"] == comparison_errors["target"])
]

print(
    "TF-IDF errors corrected by DistilBERT:",
    len(transformer_fixed)
)

transformer_fixed.head(15)

TF-IDF errors corrected by DistilBERT: 137


,text,target,tfidf_prediction,distilbert_prediction
4,Just got sent this photo from Ruby #Alaska as ...,1,0,1
26,Was in NYC last week!,0,1,0
41,on the outside you're ablaze and alive\nbut yo...,0,1,0
44,I wanted to set Chicago ablaze with my preachi...,0,1,0
107,'Nobody remembers who came in second.' Charles...,0,1,0
140,@AlexAllTimeLow awwww they're on an airplane a...,1,0,1
431,i be on that hotboy shit,0,1,0
469,@CaIxxum5SOS thanks for the damn heart attack,0,1,0
548,CIVIL WAR GENERAL BATTLE BULL RUN HERO COLONEL...,1,0,1
583,#frontpage: #Bioterror lab faced secret sancti...,0,1,0


In [45]:
tfidf_fixed = comparison_errors[
    (comparison_errors["tfidf_prediction"] == comparison_errors["target"]) &
    (comparison_errors["distilbert_prediction"] != comparison_errors["target"])
]

print(
    "TF-IDF correct but DistilBERT wrong:",
    len(tfidf_fixed)
)

TF-IDF correct but DistilBERT wrong: 82


In [47]:
tfidf_metrics_final = {
    "accuracy": accuracy_score(val_data["target"], tfidf_predictions),
    "precision": precision_score(val_data["target"], tfidf_predictions),
    "recall": recall_score(val_data["target"], tfidf_predictions),
    "f1": f1_score(val_data["target"], tfidf_predictions)
}

print("TF-IDF Validation Results")
print("-" * 35)

for metric, value in tfidf_metrics_final.items():
    print(f"{metric.capitalize():10}: {value:.4f}")

TF-IDF Validation Results
-----------------------------------
Accuracy  : 0.7967
Precision : 0.7795
Recall    : 0.7530
F1        : 0.7660


In [48]:
model_comparison = pd.DataFrame({
    "Model": [
        "TF-IDF + Balanced Logistic Regression",
        "DistilBERT"
    ],
    "Accuracy": [
        tfidf_metrics_final["accuracy"],
        transformer_metrics["eval_accuracy"]
    ],
    "Precision": [
        tfidf_metrics_final["precision"],
        transformer_metrics["eval_precision"]
    ],
    "Recall": [
        tfidf_metrics_final["recall"],
        transformer_metrics["eval_recall"]
    ],
    "F1": [
        tfidf_metrics_final["f1"],
        transformer_metrics["eval_f1"]
    ]
})

model_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1
0,TF-IDF + Balanced Logistic Regression,0.7967,0.7795,0.7530,0.7660
1,DistilBERT,0.8327,0.8420,0.7648,0.8016


In [49]:
full_train_dataset = DisasterTweetDataset(
    texts=train_df["text"],
    labels=train_df["target"],
    tokenizer=tokenizer,
    max_length=128
)

print("Full training samples:", len(full_train_dataset))

Full training samples: 7613


In [50]:
class DisasterTweetTestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

In [51]:
test_dataset = DisasterTweetTestDataset(
    texts=test_df["text"],
    tokenizer=tokenizer,
    max_length=128
)

print("Test samples:", len(test_dataset))

Test samples: 3263


In [52]:
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

final_model.to("cuda")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [53]:
final_training_args = TrainingArguments(
    output_dir="/kaggle/working/final-distilbert",

    num_train_epochs=2,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    save_strategy="no",
    logging_steps=50,

    report_to="none",

    seed=SEED
)

In [54]:
final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_train_dataset
)

final_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,1.103253
100,0.838739
150,0.764624
200,0.840537
250,0.772422
300,0.645832
350,0.694198
400,0.683020
450,0.638013


TrainOutput(global_step=476, training_loss=0.7688391388965254, metrics={'train_runtime': 120.0269, 'train_samples_per_second': 126.855, 'train_steps_per_second': 3.966, 'total_flos': 504237152984064.0, 'train_loss': 0.7688391388965254, 'epoch': 2.0})

In [55]:
test_output = final_trainer.predict(test_dataset)

test_predictions = np.argmax(
    test_output.predictions,
    axis=-1
)

print("Predictions:", len(test_predictions))
print("Non-disaster:", (test_predictions == 0).sum())
print("Disaster:", (test_predictions == 1).sum())

Predictions: 3263
Non-disaster: 2027
Disaster: 1236


In [56]:
submission = sample_submission.copy()

submission["target"] = test_predictions

submission.head()

,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1


In [57]:
print("Submission shape:", submission.shape)
print(submission["target"].value_counts())
print("Missing values:", submission.isnull().sum().sum())

Submission shape: (3263, 2)
target
0    2027
1    1236
Name: count, dtype: int64
Missing values: 0


In [58]:
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("submission.csv created successfully.")

submission.csv created successfully.


# Experiment V2 — DistilBERT with Keyword + Tweet Text

V1 uses tweet text only. V2 adds the provided keyword to the model input while keeping the validation split and training configuration unchanged.

In [126]:
print("Missing keywords:", train_df["keyword"].isna().sum())
print("Unique keywords:", train_df["keyword"].nunique())

train_df[["keyword", "text", "target"]].head(10)

Missing keywords: 61
Unique keywords: 221


,keyword,text,target
0,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,NaN,Forest fire near La Ronge Sask. Canada,1
2,NaN,All residents asked to 'shelter in place' are ...,1
3,NaN,"13,000 people receive #wildfires evacuation or...",1
4,NaN,Just got sent this photo from Ruby #Alaska as ...,1
5,NaN,#RockyFire Update => California Hwy. 20 closed...,1
6,NaN,#flood #disaster Heavy rain causes flash flood...,1
7,NaN,I'm on top of the hill and I can see a fire in...,1
8,NaN,There's an emergency evacuation happening now ...,1
9,NaN,I'm afraid that the tornado is coming to our a...,1


In [130]:
def clean_keyword(keyword):
    if pd.isna(keyword):
        return ""
    
    return str(keyword).replace("%20", " ").strip()

In [131]:
print(clean_keyword("buildings%20burning"))
print(clean_keyword("airplane%20accident"))
print(clean_keyword(np.nan))

buildings burning
airplane accident



In [133]:
def build_transformer_text(df):
    texts = []

    for keyword, text in zip(df["keyword"], df["text"]):
        keyword = clean_keyword(keyword)
        text = str(text)

        if keyword:
            combined = keyword + " [SEP] " + text
        else:
            combined = text

        texts.append(combined)

    return pd.Series(texts, index=df.index)

In [135]:
train_data["transformer_text_v2"] = build_transformer_text(train_data)
val_data["transformer_text_v2"] = build_transformer_text(val_data)

In [136]:
train_data[
    ["keyword", "text", "transformer_text_v2", "target"]
].head(10)

,keyword,text,transformer_text_v2,target
1,NaN,Forest fire near La Ronge Sask. Canada,Forest fire near La Ronge Sask. Canada,1
2,NaN,All residents asked to 'shelter in place' are ...,All residents asked to 'shelter in place' are ...,1
3,NaN,"13,000 people receive #wildfires evacuation or...","13,000 people receive #wildfires evacuation or...",1
5,NaN,#RockyFire Update => California Hwy. 20 closed...,#RockyFire Update => California Hwy. 20 closed...,1
6,NaN,#flood #disaster Heavy rain causes flash flood...,#flood #disaster Heavy rain causes flash flood...,1
8,NaN,There's an emergency evacuation happening now ...,There's an emergency evacuation happening now ...,1
10,NaN,Three people died from the heat wave so far,Three people died from the heat wave so far,1
11,NaN,Haha South Tampa is getting flooded hah- WAIT ...,Haha South Tampa is getting flooded hah- WAIT ...,1
12,NaN,#raining #flooding #Florida #TampaBay #Tampa 1...,#raining #flooding #Florida #TampaBay #Tampa 1...,1
13,NaN,#Flood in Bago Myanmar #We arrived Bago,#Flood in Bago Myanmar #We arrived Bago,1


In [137]:
train_data.loc[
    train_data["keyword"].notna(),
    ["keyword", "text", "transformer_text_v2", "target"]
].head(10)

,keyword,text,transformer_text_v2,target
31,ablaze,@bbcmtd Wholesale Markets ablaze http://t.co/l...,ablaze [SEP] @bbcmtd Wholesale Markets ablaze ...,1
32,ablaze,We always try to bring the heavy. #metal #RT h...,ablaze [SEP] We always try to bring the heavy....,0
33,ablaze,#AFRICANBAZE: Breaking news:Nigeria flag set a...,ablaze [SEP] #AFRICANBAZE: Breaking news:Niger...,1
39,ablaze,Ablaze for you Lord :D,ablaze [SEP] Ablaze for you Lord :D,0
40,ablaze,Check these out: http://t.co/rOI2NSmEJJ http:/...,ablaze [SEP] Check these out: http://t.co/rOI2...,0
42,ablaze,Had an awesome time visiting the CFC head offi...,ablaze [SEP] Had an awesome time visiting the ...,0
45,ablaze,I gained 3 followers in the last week. You? Kn...,ablaze [SEP] I gained 3 followers in the last ...,0
47,ablaze,Building the perfect tracklist to life leave t...,ablaze [SEP] Building the perfect tracklist to...,0
48,ablaze,Check these out: http://t.co/rOI2NSmEJJ http:/...,ablaze [SEP] Check these out: http://t.co/rOI2...,0
49,ablaze,First night with retainers in. It's quite weir...,ablaze [SEP] First night with retainers in. It...,0


In [138]:
train_data.loc[
    train_data["keyword"].astype(str).str.contains("%20"),
    ["keyword", "transformer_text_v2"]
].head(10)

,keyword,transformer_text_v2
136,airplane%20accident,airplane accident [SEP] Experts in France begi...
138,airplane%20accident,airplane accident [SEP] @crobscarla your lifet...
141,airplane%20accident,airplane accident [SEP] family members of osam...
142,airplane%20accident,airplane accident [SEP] Man Goes into Airplane...
143,airplane%20accident,airplane accident [SEP] Horrible Accident Man...
144,airplane%20accident,airplane accident [SEP] A Cessna airplane acci...
145,airplane%20accident,airplane accident [SEP] #Horrible #Accident Ma...
146,airplane%20accident,airplane accident [SEP] Experts in France begi...
147,airplane%20accident,airplane accident [SEP] Experts in France begi...
148,airplane%20accident,airplane accident [SEP] #KCA #VoteJKT48ID mbat...


In [139]:
v2_token_lengths = train_data["transformer_text_v2"].apply(
    lambda text: len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
)

print(v2_token_lengths.describe())

print("\nMaximum token length:", v2_token_lengths.max())
print("Tweets longer than 128:", (v2_token_lengths > 128).sum())

count    6083.000000
mean       35.683051
std        12.323152
min         4.000000
25%        28.000000
50%        36.000000
75%        44.000000
max        84.000000
Name: transformer_text_v2, dtype: float64

Maximum token length: 84
Tweets longer than 128: 0


In [140]:
print("Training rows:", len(train_data))
print("With keyword:", train_data["keyword"].notna().sum())
print("Without keyword:", train_data["keyword"].isna().sum())

Training rows: 6083
With keyword: 6037
Without keyword: 46


In [142]:
train_dataset_v2 = DisasterTweetDataset(
    texts=train_data["transformer_text_v2"],
    labels=train_data["target"],
    tokenizer=tokenizer,
    max_length=128
)

val_dataset_v2 = DisasterTweetDataset(
    texts=val_data["transformer_text_v2"],
    labels=val_data["target"],
    tokenizer=tokenizer,
    max_length=128
)

print("V2 training samples:", len(train_dataset_v2))
print("V2 validation samples:", len(val_dataset_v2))

V2 training samples: 6083
V2 validation samples: 1530


In [143]:
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model_v2.to("cuda")

print("V2 model device:", next(model_v2.parameters()).device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


V2 model device: cuda:0


In [144]:
training_args_v2 = TrainingArguments(
    output_dir="/kaggle/working/distilbert-keyword-v2",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",

    seed=SEED
)

In [145]:
trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_dataset_v2,
    eval_dataset=val_dataset_v2,
    compute_metrics=compute_metrics
)

In [146]:
trainer_v2.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.833076,0.821602,0.826797,0.837438,0.754438,0.793774
2,0.710471,0.838775,0.830719,0.828346,0.778107,0.802441
3,0.570224,0.859091,0.826797,0.826709,0.769231,0.796935


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=573, training_loss=0.7167582154065942, metrics={'train_runtime': 161.6953, 'train_samples_per_second': 112.86, 'train_steps_per_second': 3.544, 'total_flos': 604349389518336.0, 'train_loss': 0.7167582154065942, 'epoch': 3.0})

In [147]:
v2_metrics = trainer_v2.evaluate()

print("DistilBERT V2 — Keyword + Text")
print("-" * 40)

print(f"Accuracy : {v2_metrics['eval_accuracy']:.4f}")
print(f"Precision: {v2_metrics['eval_precision']:.4f}")
print(f"Recall   : {v2_metrics['eval_recall']:.4f}")
print(f"F1 Score : {v2_metrics['eval_f1']:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


DistilBERT V2 — Keyword + Text
----------------------------------------
Accuracy : 0.8307
Precision: 0.8283
Recall   : 0.7781
F1 Score : 0.8024


In [148]:
v2_output = trainer_v2.predict(val_dataset_v2)

v2_predictions = np.argmax(
    v2_output.predictions,
    axis=-1
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [149]:
v2_cm = confusion_matrix(
    val_data["target"],
    v2_predictions
)

v2_cm_df = pd.DataFrame(
    v2_cm,
    index=["Actual Non-Disaster", "Actual Disaster"],
    columns=["Predicted Non-Disaster", "Predicted Disaster"]
)

v2_cm_df

,Predicted Non-Disaster,Predicted Disaster
Actual Non-Disaster,745,109
Actual Disaster,150,526


In [150]:
v1_v2_comparison = pd.DataFrame({
    "Model": [
        "V1 - DistilBERT Text Only",
        "V2 - DistilBERT Keyword + Text"
    ],
    "Accuracy": [
        transformer_metrics["eval_accuracy"],
        v2_metrics["eval_accuracy"]
    ],
    "Precision": [
        transformer_metrics["eval_precision"],
        v2_metrics["eval_precision"]
    ],
    "Recall": [
        transformer_metrics["eval_recall"],
        v2_metrics["eval_recall"]
    ],
    "F1": [
        transformer_metrics["eval_f1"],
        v2_metrics["eval_f1"]
    ]
})

v1_v2_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1
0,V1 - DistilBERT Text Only,0.8327,0.8420,0.7648,0.8016
1,V2 - DistilBERT Keyword + Text,0.8307,0.8283,0.7781,0.8024


In [151]:
import torch

v2_logits = torch.tensor(v2_output.predictions)

v2_probs = torch.softmax(
    v2_logits,
    dim=1
)[:, 1].numpy()

print(v2_probs[:10])

[0.8268746  0.95151913 0.4520513  0.96273845 0.9828834  0.0800816
 0.10264205 0.09489465 0.06154574 0.08269029]


In [152]:
threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):
    preds = (v2_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(val_data["target"], preds),
        "Precision": precision_score(val_data["target"], preds),
        "Recall": recall_score(val_data["target"], preds),
        "F1": f1_score(val_data["target"], preds)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(
    "F1",
    ascending=False
).head(10).round(4)

,Threshold,Accuracy,Precision,Recall,F1
34,0.54,0.8346,0.8395,0.7737,0.8052
19,0.39,0.8281,0.8068,0.8033,0.8050
33,0.53,0.8340,0.8371,0.7751,0.8049
15,0.35,0.8248,0.7931,0.8166,0.8047
21,0.41,0.8281,0.8087,0.8003,0.8045
18,0.38,0.8268,0.8027,0.8062,0.8044
20,0.40,0.8275,0.8065,0.8018,0.8042
17,0.37,0.8255,0.7985,0.8092,0.8038
35,0.55,0.8340,0.8414,0.7692,0.8037
16,0.36,0.8242,0.7937,0.8136,0.8035


In [153]:
best_threshold_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

best_threshold_row

Threshold    0.540000
Accuracy     0.834641
Precision    0.839486
Recall       0.773669
F1           0.805235
Name: 34, dtype: float64

In [154]:
best_threshold = best_threshold_row["Threshold"]

print(f"Best threshold: {best_threshold:.2f}")
print(f"Accuracy : {best_threshold_row['Accuracy']:.4f}")
print(f"Precision: {best_threshold_row['Precision']:.4f}")
print(f"Recall   : {best_threshold_row['Recall']:.4f}")
print(f"F1 Score : {best_threshold_row['F1']:.4f}")

Best threshold: 0.54
Accuracy : 0.8346
Precision: 0.8395
Recall   : 0.7737
F1 Score : 0.8052


In [155]:
default_row = threshold_df.iloc[
    (threshold_df["Threshold"] - 0.50).abs().argsort()[:1]
]

print("Default threshold:")
display(default_row.round(4))

print("\nBest threshold:")
display(
    pd.DataFrame([best_threshold_row]).round(4)
)

Default threshold:


,Threshold,Accuracy,Precision,Recall,F1
30,0.5,0.8307,0.8283,0.7781,0.8024



Best threshold:


,Threshold,Accuracy,Precision,Recall,F1
34,0.54,0.8346,0.8395,0.7737,0.8052


In [156]:
v2_tuned_predictions = (
    v2_probs >= best_threshold
).astype(int)

v2_tuned_cm = confusion_matrix(
    val_data["target"],
    v2_tuned_predictions
)

v2_tuned_cm_df = pd.DataFrame(
    v2_tuned_cm,
    index=["Actual Non-Disaster", "Actual Disaster"],
    columns=["Predicted Non-Disaster", "Predicted Disaster"]
)

v2_tuned_cm_df

,Predicted Non-Disaster,Predicted Disaster
Actual Non-Disaster,754,100
Actual Disaster,153,523


In [157]:
print(classification_report(
    val_data["target"],
    v2_tuned_predictions,
    target_names=["Non-Disaster", "Disaster"]
))

              precision    recall  f1-score   support

Non-Disaster       0.83      0.88      0.86       854
    Disaster       0.84      0.77      0.81       676

    accuracy                           0.83      1530
   macro avg       0.84      0.83      0.83      1530
weighted avg       0.83      0.83      0.83      1530



In [158]:
train_df["transformer_text_v2"] = build_transformer_text(train_df)
test_df["transformer_text_v2"] = build_transformer_text(test_df)

print(train_df[
    ["keyword", "text", "transformer_text_v2"]
].head())

print("\nFull training samples:", len(train_df))
print("Test samples:", len(test_df))

  keyword                                               text  \
0     NaN  Our Deeds are the Reason of this #earthquake M...   
1     NaN             Forest fire near La Ronge Sask. Canada   
2     NaN  All residents asked to 'shelter in place' are ...   
3     NaN  13,000 people receive #wildfires evacuation or...   
4     NaN  Just got sent this photo from Ruby #Alaska as ...   

                                 transformer_text_v2  
0  Our Deeds are the Reason of this #earthquake M...  
1             Forest fire near La Ronge Sask. Canada  
2  All residents asked to 'shelter in place' are ...  
3  13,000 people receive #wildfires evacuation or...  
4  Just got sent this photo from Ruby #Alaska as ...  

Full training samples: 7613
Test samples: 3263


In [159]:
full_train_dataset_v2 = DisasterTweetDataset(
    texts=train_df["transformer_text_v2"],
    labels=train_df["target"],
    tokenizer=tokenizer,
    max_length=128
)

test_dataset_v2 = DisasterTweetTestDataset(
    texts=test_df["transformer_text_v2"],
    tokenizer=tokenizer,
    max_length=128
)

print("Full V2 train:", len(full_train_dataset_v2))
print("V2 test:", len(test_dataset_v2))

Full V2 train: 7613
V2 test: 3263


In [160]:
final_model_v2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

final_model_v2.to("cuda")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [161]:
final_args_v2 = TrainingArguments(
    output_dir="/kaggle/working/final-distilbert-v2",

    num_train_epochs=2,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    save_strategy="no",
    logging_steps=50,
    report_to="none",

    seed=SEED
)

final_trainer_v2 = Trainer(
    model=final_model_v2,
    args=final_args_v2,
    train_dataset=full_train_dataset_v2
)

final_trainer_v2.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,1.114427
100,0.838810
150,0.764414
200,0.841804
250,0.749233
300,0.674960
350,0.729790
400,0.675607
450,0.631889


TrainOutput(global_step=476, training_loss=0.7740085705989549, metrics={'train_runtime': 120.2736, 'train_samples_per_second': 126.595, 'train_steps_per_second': 3.958, 'total_flos': 504237152984064.0, 'train_loss': 0.7740085705989549, 'epoch': 2.0})

In [162]:
final_test_output_v2 = final_trainer_v2.predict(
    test_dataset_v2
)

final_test_logits_v2 = torch.tensor(
    final_test_output_v2.predictions
)

final_test_probs_v2 = torch.softmax(
    final_test_logits_v2,
    dim=1
)[:, 1].numpy()

In [168]:
FINAL_THRESHOLD = float(best_threshold)

print(f"Using validation-selected threshold: {FINAL_THRESHOLD:.2f}")

final_test_predictions_v2 = (
    final_test_probs_v2 >= FINAL_THRESHOLD
).astype(int)

print("Predictions:", len(final_test_predictions_v2))
print("Non-disaster:", (final_test_predictions_v2 == 0).sum())
print("Disaster:", (final_test_predictions_v2 == 1).sum())

Using validation-selected threshold: 0.54
Predictions: 3263
Non-disaster: 2044
Disaster: 1219


In [169]:
submission_v2_tuned = sample_submission.copy()

submission_v2_tuned["target"] = final_test_predictions_v2

print(submission_v2_tuned.head())
print()
print("Prediction distribution:")
print(submission_v2_tuned["target"].value_counts())

assert len(submission_v2_tuned) == 3263
assert submission_v2_tuned["target"].isna().sum() == 0
assert set(submission_v2_tuned["target"].unique()).issubset({0, 1})

submission_v2_tuned.to_csv(
    "/kaggle/working/submission_v2_tuned.csv",
    index=False
)

print("\nsubmission_v2_tuned.csv created successfully.")

   id  target
0   0       1
1   2       1
2   3       1
3   9       1
4  11       1

Prediction distribution:
target
0    2044
1    1219
Name: count, dtype: int64

submission_v2_tuned.csv created successfully.


## Final Model Comparison and Conclusion

### Classical Baseline
A TF-IDF representation with class-balanced Logistic Regression was used as the classical NLP baseline.

- Accuracy: 0.7987
- Precision: 0.7822
- Recall: 0.7544
- F1 Score: 0.7681

### DistilBERT V1 — Tweet Text Only
Fine-tuning DistilBERT on tweet text substantially improved performance over the classical baseline.

- Accuracy: 0.8327
- Precision: 0.8420
- Recall: 0.7648
- F1 Score: 0.8016
- Kaggle Public Score: 0.83757

### DistilBERT V2 — Keyword + Tweet Text
Keyword metadata was added to the tweet text while keeping the validation split and training configuration unchanged.

- Accuracy: 0.8307
- Precision: 0.8283
- Recall: 0.7781
- F1 Score: 0.8024

The keyword feature slightly increased validation F1 and recall, but the improvement over V1 was marginal.

### Threshold Tuning
Decision thresholds from 0.20 to 0.80 were evaluated on the held-out validation set. A threshold of 0.54 produced the highest F1 in the final V2 experiment.

- Accuracy: 0.8346
- Precision: 0.8395
- Recall: 0.7737
- F1 Score: 0.8052

Although threshold tuning improved held-out validation performance, it did not improve the Kaggle public leaderboard result.

### Final Selection
The text-only DistilBERT V1 submission achieved the highest Kaggle public score of **0.83757** and was retained as the final Kaggle submission.

The experiments show that the Transformer approach clearly outperformed the classical TF-IDF baseline. Keyword metadata and threshold tuning provided small improvements on the held-out validation set, but those improvements did not generalize to a higher public leaderboard score.